# Scaled Dot-Product Attention: Step-by-Step Implementation

This notebook walks through the progressive implementation of the attention mechanism, starting from the simplest possible version and gradually adding complexity to approach a more complete transformer-style attention block.

$$
\text{Attention} = \text{softmax}\left(\frac{QK^T}{\sqrt{d_{model}}}\right) V
$$

The levels of complexity are described below:
- basic implementation
- \+ batch support
- \+ multi head attention
- \+ masking
- \+ dropout

In [1]:
import numpy as np

In [54]:
# embedding size of 512, 
# total tokens of 10

d_model = 512
seq_len = 10

In [55]:
# Random inputs solely for the forward pass (inputs don't matter since no backward pass - they are just here to exist)
x_input = np.random.randn(seq_len, d_model)

## Basic Implementation:
The following implementation provides a **basic, non-batched** version of **scaled dot-product attention**, capturing the essential computation:
- Linear projections for queries ($Q$), keys ($K$), and values ($V$)
- Dot product between $Q$ and $K$
- Scaling by $\sqrt{d_k}$ for numerical stability
- Softmax normalization of attention scores
- Final output as a weighted sum of value vectors

This version is intended to demonstrate the **core mechanics** without additional features like batching, masking, dropout, or multi-head attention. These will be added step-by-step in later sections.


In [56]:
class Attention:
    def __init__(self, d_k):
        self.v = np.random.randn(d_k, d_k)
        self.q = np.random.randn(d_k, d_k)
        self.k = np.random.randn(d_k, d_k)
        self.d_k = d_k

    def softmax(self, x, axis=None):
        x_shifted = x - x.max(axis=axis, keepdims=True) # for numerical stability
        return np.exp(x_shifted) / np.exp(x_shifted).sum(axis=axis, keepdims=True)
    
    def forward(self, X):
        # Linear projections
        Q = X @ self.q
        K = X @ self.k
        V = X @ self.v

        # Scaled dot product
        QK = Q @ K.T
        div_term = self.d_k ** 0.5
        QK_scaled = QK / div_term

        # Attention between tokens
        self.attention = self.softmax(QK_scaled, axis=1)

        # Final projection
        output = self.attention @ V

        return output

    def __call__(self, X):
        return self.forward(X)

In [57]:
attn = Attention(512)

In [58]:
attn(x_input).shape, attn.attention.shape

((10, 512), (10, 10))

## Adding Batch Support:

This section extends the basic attention implementation to support **batched input**.

The input tensor now follows the standard shape:

`(batch_size, sequence_length, d_model)`

Key updates include:

- Supporting batch-wise computation of $Q$, $K$, and $V$
- Computing attention scores for each sequence in the batch simultaneously
- Ensuring softmax is applied correctly across the sequence dimension within each batch
- Maintaining output shape consistency: `(batch_size, sequence_length, d_model)`

This implementation is functionally equivalent to the basic implementation but generalized to operate on multiple sequences in parallel.

In [59]:
# embedding size of 512, 
# total tokens of 10

d_model = 512
seq_len = 10
batch_size = 5

In [60]:
# Random inputs solely for the forward pass (inputs don't matter since no backward pass - they are just here to exist)
x_input = np.random.randn(batch_size, seq_len, d_model)

In [61]:
tmp_matrix = np.random.randn(d_model, 2)
(x_input @ tmp_matrix).shape

(5, 10, 2)

It seems like python's `@` already supports this, so why do we need to build out a separate implementaiton accounting for batches?

In [62]:
attn = Attention(512)
attn(x_input)

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 10 is different from 512)

We see this is an error with the Q @ K.T multiplication, what happens is we do `Q @ K.T`. For this to work, 

- Q Should be `[batch_size, input_size, d_model]`
- K Should be `[batch_size, input_size, d_model]`
- K.T Should be `[batch_size, d_model, input_size]`

However, if we take the transpose of a 3d array:

In [63]:
x_input.shape, x_input.T.shape

((5, 10, 512), (512, 10, 5))

We see that the entire order has flipped, we need `batch_size` to stay in the first position for python's matrix multiplication to work

In [64]:
np.transpose(x_input, (0,2,1)).shape

(5, 512, 10)

This worked, let's see how the rest of the function should be affected

In [67]:
tmp_matrix = np.array(
    ([
        [1,1,1],
        [1,2,3],
        [3,3,3],
    ],
    [
        [3,2,1],
        [2,4,6],
        [10,5,1]
    ])
)

print('With axis=1:\n', attn.softmax(tmp_matrix, axis=1))
print()
print('With axis=2:\n', attn.softmax(tmp_matrix, axis=2))

With axis=1:
 [[[1.06506979e-01 9.00305732e-02 6.33789383e-02]
  [1.06506979e-01 2.44728471e-01 4.68310531e-01]
  [7.86986042e-01 6.65240956e-01 4.68310531e-01]]

 [[9.10745952e-04 3.51190270e-02 6.64835448e-03]
  [3.35044712e-04 2.59496460e-01 9.86703291e-01]
  [9.98754209e-01 7.05384513e-01 6.64835448e-03]]]

With axis=2:
 [[[3.33333333e-01 3.33333333e-01 3.33333333e-01]
  [9.00305732e-02 2.44728471e-01 6.65240956e-01]
  [3.33333333e-01 3.33333333e-01 3.33333333e-01]]

 [[6.65240956e-01 2.44728471e-01 9.00305732e-02]
  [1.58762400e-02 1.17310428e-01 8.66813332e-01]
  [9.93185401e-01 6.69203059e-03 1.22568816e-04]]]


We will need to update the axis for softmax, otherwise it will do the columns of the attention matrix for each batch. Since the rowwise sum corresponds to the 3rd dimentions, we need to set `axis=2`.

The rest of the function should be fine as we have no more transposes or dimentions issues

In [68]:
class Attention:
    def __init__(self, d_k):
        self.v = np.random.randn(d_k, d_k)  # (d_k, d_k)
        self.q = np.random.randn(d_k, d_k)  # (d_k, d_k)
        self.k = np.random.randn(d_k, d_k)  # (d_k, d_k)
        self.d_k = d_k

    def softmax(self, x, axis=None):
        x_shifted = x - x.max(axis=axis, keepdims=True)  # for numerical stability
        return np.exp(x_shifted) / np.exp(x_shifted).sum(axis=axis, keepdims=True)

    def forward(self, X):
        # X: (batch_size, seq_len, d_k)

        # Linear projections
        Q = X @ self.q  # [batch_size, seq_len, d_k]
        K = X @ self.k  # [batch_size, seq_len, d_k]
        V = X @ self.v  # [batch_size, seq_len, d_k]

        # Scaled dot product
        QK = Q @ np.transpose(K, (0, 2, 1))  # [batch_size, seq_len, seq_len]
        div_term = self.d_k ** 0.5
        QK_scaled = QK / div_term  # [batch_size, seq_len, seq_len]

        # Attention weights
        self.attention = self.softmax(QK_scaled, axis=2)  # [batch_size, seq_len, seq_len]

        # Apply attention to values
        output = self.attention @ V  # [batch_size, seq_len, d_k]

        return output

    def __call__(self, X):
        return self.forward(X)


In [69]:
attn = Attention(512)
attn(x_input).shape

(5, 10, 512)

It worked, let's pass the same input as two different batches to see if we get the same output

In [70]:
x_input = np.random.randn(seq_len, d_model)
x_input = np.stack((x_input, x_input))
x_input.shape

(2, 10, 512)

In [71]:
output = attn(x_input)
print((output[0] == output[1]).all())

True


## Implementing Multi-Head Attention:

In this section, we enhance the attention mechanism by implementing **multi-head attention**, which allows the model to learn many different patterns *(similar to having multiple filters in a convolutional neural network)*.

Key concepts introduced here:

- Splitting the input projections ($Q$, $K$, and $V$) into multiple heads along the feature dimension.
- Performing scaled dot-product attention independently for each head in parallel.
- Concatenating the outputs of all heads back into a single tensor.
- Applying a final linear projection to combine the multi-head outputs into the original feature dimension.

Multi-head attention increases the model’s ability to capture diverse aspects of the input sequences and has become a standard building block in transformer architectures.

This implementation assumes batched inputs and builds directly on the batching functionality added in the previous section.